In [49]:
# =================================
# 設定
# =================================
IS_ONLINE_JUDGE = False

LIMIT_QUERY_CNT = None

DEBUG = True
DEBUG_QUERY = None
DEBUG_L = None

MAX_TIME = 1.25
FILE_NUM = 100

# =================================
# 初期化
# =================================
import time

GLOBAL_START_TIME = time.perf_counter()

import random

random.seed(0)

import sys

sys.setrecursionlimit(10**9)
if IS_ONLINE_JUDGE:
    DEBUG = False

# =================================
# Import
# =================================
import math

# =================================
# 共通
# =================================
if True:
    from importlib import reload

    import src.common as common

    reload(common)


from src.common import (
    Env,
    EnvOffline,
    calc_dist,
    calc_rectangle_dist_min_max,
    construct_dag_and_rdag_from_edges,
    construct_dag_from_edges,
    construct_dist_matrix,
    construct_dist_matrix_rectangle,
    construct_graph,
    construct_sorted_edges,
    cut_graph,
    divide_interval,
    exponential_schedule,
    is_overlap,
    kruskals_algorithm,
    linear_schedule,
    my_deepcopy,
    prim,
    prim_k,
    prim_vs,
    priority_topological_sort,
    scc,
    scc_construct,
    sort_pair,
)


# =================================
# 汎用（main.py専用）
# =================================
def debug_print(*args, **kwargs):
    if IS_ONLINE_JUDGE:
        return
    print(*args, **kwargs)


def print_elapsed_time():
    debug_print(f"elapsed time: {(time.perf_counter() - GLOBAL_START_TIME) * 1000:.2f}msecs")

## クエリによる順位更新

- 制約リスト <= []
- Q 回繰り返す

  - ランダムな点で MST を得る
  - MST の 1 辺を選ぶ
    - 辺をカットする
    - 2 グループを得る
    - グループ間の全ての辺を貼る（ - 元の辺）
    - 元の辺 < 抽出した辺の制約をリストに追加する

- トポロジカルソートしながら元の順位を保持して辺をソートする
- error を計算してみる


In [50]:
def calc_edge_list(env, points):
    edge_dists = []
    edge_id = 0
    for i in range(env.N):
        for j in range(i + 1, env.N):
            dist = calc_dist(points[i], points[j])
            edge_dists.append((dist, edge_id))
            edge_id += 1
    return sorted(edge_dists, key=lambda x: x[0])


def rank_sqrt_error(rank1, rank2):
    N = len(rank1)
    rank2_num_to_rank = {}
    for i, r2 in enumerate(rank2):
        rank2_num_to_rank[r2] = i

    cost = 0
    for i, v in enumerate(rank1):
        cost += abs(i - rank2_num_to_rank[v]) ** 2

    return cost / N


def rank_error(rank1, rank2):
    N = len(rank1)
    rank2_num_to_rank = {}
    for i, r2 in enumerate(rank2):
        rank2_num_to_rank[r2] = i

    cost = 0
    for i, v in enumerate(rank1):
        cost += abs(i - rank2_num_to_rank[v])

    return cost / N


def sorted_dag(
    vs: list[int],  # 頂点のリスト
    graph: dict[int, list[int]],  # 隣接リスト
    graph_r: dict[int, list[int]],  # 逆グラフ
    initial_ranking: list[int],  # 初期のランク
):
    """閉路を含むDAGを初期のランクに従ってソートする"""
    label_num, group = scc(graph, graph_r, vs)
    ssc_graph, belong_lists = scc_construct(graph, label_num, group, vs)

    num_to_ranking = {i: j for j, i in enumerate(initial_ranking)}
    construct_initial_ranking = {}
    for i, belong_list in enumerate(belong_lists):
        ssc_ranks = [num_to_ranking[v] for v in belong_list]
        n = len(ssc_ranks)
        mid_rank = sorted(ssc_ranks)[n // 2]
        construct_initial_ranking[i] = mid_rank

    result_rank_construct = priority_topological_sort(ssc_graph, construct_initial_ranking)
    result_rank = []
    for i in result_rank_construct:
        now_belong = belong_lists[i]
        result_rank.extend(now_belong)

    return result_rank

In [51]:
file_num = 0
input_file_path = f"../in/{file_num:04d}.txt"
output_file_path = f"../out/{file_num:04d}.txt"
env = EnvOffline(input_file_path, output_file_path)

edgeid_to_poinid = {}
pointid_to_edgeid = {}
edge_id = 0
for i in range(env.N):
    for j in range(i + 1, env.N):
        edgeid_to_poinid[edge_id] = (i, j)
        pointid_to_edgeid[(i, j)] = edge_id
        edge_id += 1

correct_edge_dists = calc_edge_list(env, env.coordinates)
estimate_edge_dists = calc_edge_list(env, env.cneter_points)

correct_ranks = [v[1] for v in correct_edge_dists]
estimate_ranks = [v[1] for v in estimate_edge_dists]

EDGE_NUM = len(estimate_ranks)

In [52]:
constrains = set()

In [53]:
print(EDGE_NUM)
400 * 400

319600


160000

In [54]:
constrains = set()

for i in range(EDGE_NUM):
    for j in range(1, 51):
        nj = i + j**2
        if nj >= EDGE_NUM:
            break
        ni = estimate_ranks[i]
        nj = estimate_ranks[nj]
        pi = edgeid_to_poinid[ni]
        pj = edgeid_to_poinid[nj]

        r01, r02 = env.rectangles[pi[0]], env.rectangles[pi[1]]
        r11, r12 = env.rectangles[pj[0]], env.rectangles[pj[1]]
        dmin1, dmax1 = calc_rectangle_dist_min_max(r01, r02)
        dmin2, dmax2 = calc_rectangle_dist_min_max(r11, r12)
        # print(
        #     dmin1,
        #     dmax1,
        #     dmin2,
        #     dmax2,
        #     ni,
        #     nj,
        #     pi,
        #     pj,
        #     env.rectangles[pi[0]],
        #     env.rectangles[pi[1]],
        #     env.rectangles[pj[0]],
        #     env.rectangles[pj[1]],
        # )
        if is_overlap(dmin1, dmax1, dmin2, dmax2):
            continue
        elif dmax1 < dmin2:
            constrains.add((ni, nj))
        elif dmax2 < dmin1:
            constrains.add((nj, ni))
        else:
            debug_print("error")

        # c11 = env.coordinates[pi[0]]
        # c12 = env.coordinates[pi[1]]
        # c21 = env.coordinates[pj[0]]
        # c22 = env.coordinates[pj[1]]
        # if calc_dist(c11, c12) <= calc_dist(c21, c22):
        #     constrains.add((ni, nj))
        # elif calc_dist(c11, c12) > calc_dist(c21, c22):
        #     constrains.add((nj, ni))

In [55]:
print(len(constrains))

2897


In [56]:
QUERY_ROOP = 10**5
# divide_interval(0, EDGE_NUM, QUERY_ROOP + 2)

In [ ]:
for qi in range(QUERY_ROOP):
    point_ids = list(range(env.N))
    random_vs = random.sample(point_ids, k=15)
    edges = env.query(random_vs)

    graph = construct_dag_from_edges(edges, random_vs)

    for e in edges:
        e1_id = pointid_to_edgeid[e]
        group_vs1, group_vs2 = cut_graph(graph, e)
        for v1 in group_vs1:
            for v2 in group_vs2:
                sv1, sv2 = sort_pair((v1, v2))
                if e == (sv1, sv2):
                    continue
                e2_id = pointid_to_edgeid[(sv1, sv2)]
                constrains.add((e1_id, e2_id))

                if DEBUG:
                    long_c1 = env.coordinates[sv1]
                    long_c2 = env.coordinates[sv2]
                    short_c1 = env.coordinates[e[0]]
                    short_c2 = env.coordinates[e[1]]
                    dist1 = calc_dist(short_c1, short_c2)
                    dist2 = calc_dist(long_c1, long_c2)
                    if dist1 > dist2:
                        debug_print((dist1, dist2), e, (sv1, sv2), e1_id, e2_id)
                    assert dist1 <= dist2

In [58]:
edge_ids = list(range(len(estimate_ranks)))
graph, graph_r = construct_dag_and_rdag_from_edges(constrains, edge_ids)
result_ranks = sorted_dag(
    edge_ids,
    graph,
    graph_r,
    estimate_ranks,
)

In [59]:
print("四角形中心", rank_error(correct_ranks, estimate_ranks))
print("制約", rank_error(correct_ranks, result_ranks))

四角形中心 8605.000425531915
制約 8477.95565707134


In [60]:
def calc_greedy_answer(env: Env, target_points):
    graph = construct_dist_matrix(target_points)

    groups = [(i, g) for i, g in enumerate(env.G)]
    groups = sorted(groups, key=lambda x: x[1], reverse=True)

    ans_edges = [None for _ in range(len(groups))]
    ans_v = [None for _ in range(len(groups))]
    ans_costs = [None for _ in range(len(groups))]

    now_used: set = set()
    not_used: set = set(range(env.N))
    for group_n, group_size in groups:
        v = not_used.pop()
        prim_edges, prim_v, prim_cost = prim_k(graph, v, group_size, now_used, env.N)
        ans_edges[group_n] = prim_edges
        ans_v[group_n] = prim_v
        ans_costs[group_n] = prim_cost
        now_used.update(prim_v)
        not_used.difference_update(prim_v)

    return ans_v, ans_edges, ans_costs